# Конструирование алгоритмов mclp

In [ ]:
import numpy as np
import networkx as nx
import pandas as pd
import geopandas as gpd
from genesis.core import BestPointsBase, MetricBase, StateBase
from genesis.best_points import BestNodeHillClimbing
from genesis.states import FirstArrivalUnitState
from genesis.metrics import ArrivalTime
from genesis.tools import timing
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import osmnx as ox

In [ ]:
# Загрузка графа и полигона границ
G = ox.load_graphml("tests/data/test_rng.ml")
area = gpd.read_file("tests/data/test_polygon.gpkg")

In [ ]:
class BestNodesGenesis(BestPointsBase):
    '''
    Поиск лучших узлов для размещения n объектов.
    '''

    def __init__(self,
                 state_function: StateBase,
                 metric_function: MetricBase,
                 best_point_function: BestPointsBase,
                 appr_val: float = 0.95,
                 iterations=5,
                 **kwargs) -> None:
        
        self.best_point_function = best_point_function
        self.appr_val = appr_val
        self.iterations = iterations
        super().__init__(state_function, metric_function, **kwargs)


    def __call__(self,
                 env:nx.MultiDiGraph,
                 dynamic_nodes: list|dict,
                 static_nodes: list|dict = None,
                 before_iters_start_function=None,
                 iter_calc_end_function=None,
                 area = None,
                 **kwargs):
        
        # Проверка корректности пришедших данных
        if not isinstance(env, nx.MultiDiGraph):
            raise TypeError("Тип аргумента `env` должен быть MultiDiGraph!")
        if not static_nodes is None:
            if not ((isinstance(dynamic_nodes,list) and isinstance(static_nodes,list)) or \
                (isinstance(dynamic_nodes,dict) and isinstance(static_nodes,dict))):
                raise TypeError("Аргументы `dynamic_nodes` и `static_nodes` должны быть одинакового типа: list|dict")

        # Если статические узлы не указаны - заменяем значение переменной с None
        # на [] или {} в зависимости от типа данных `dynamic_nodes`
        if static_nodes is None:
            if isinstance(dynamic_nodes,list):
                static_nodes = []
            elif isinstance(dynamic_nodes,dict):
                static_nodes = {}

        # 0. Получение суммарного списка (или словаря) узлов для расчета состояния и метрик
        if isinstance(dynamic_nodes,list):
            start_nodes = dynamic_nodes+static_nodes
        elif isinstance(dynamic_nodes,dict):
            start_nodes = {**dynamic_nodes, **static_nodes}

        # 1. Расчет состояния
        times, nearest = self.state_function(env=env, points=start_nodes, **kwargs)   # area=area, - убрал

        # 2. Расчет стартовой метрики состояния и определение стартового размещения
        best_metric = self.metric_function(times, area=area, **kwargs)
        best_dynamic_nodes = dynamic_nodes

        # выполняем функцию перед началом итераций
        if before_iters_start_function:
            before_iters_start_function(
                                    best_metric=best_metric,
                                    dynamic_nodes=dynamic_nodes,
                                    static_nodes=static_nodes
                                    )

        # Итерации расчета лучшего размещения
        for iteration in range(self.iterations):

            # 2. Поиск для каждого из dinamic_nodes наилучшего размещения в пределах своей зоны обслуживания
            # 2.1. Для списка:
            if isinstance(dynamic_nodes, list):
                new_dynamic_nodes = []
                for dynamic_node in dynamic_nodes:

                    # Определяем список узлов в которые из узла dynamic_node можно прибыть первым
                    node_area_nodes = [n for n,v in nearest.items() if v==dynamic_node]

                    # Определяем подграф зоны обслуживания для узла dynamic_node
                    node_area_G = nx.subgraph(env, node_area_nodes)
                    # print(dynamic_node in node_area_nodes)

                    # Определяем лучший узел
                    best_nodes, _ = self.best_point_function(env=node_area_G, 
                                                                    start_node=dynamic_node,
                                                                    # area=area,
                                                                    appr_val = self.appr_val,
                                                                    **kwargs)[:2] # [:2] Это для ограничения вывода дебаг-данных в некоторых функциях
                    if isinstance(best_nodes, list):
                        best_nodes = best_nodes[0]
                    # Добавляем полученный узел в новый список
                    new_dynamic_nodes.append(best_nodes)
            else:
                new_dynamic_nodes = {}
                for dynamic_node_id, dynamic_node_key in dynamic_nodes.items():

                    # Определяем список узлов в которые из узла dynamic_node можно прибыть первым
                    node_area_nodes = [k for k,v in nearest.items() if v==dynamic_node_key]

                    # Определяем подграф зоны обслуживания для узла dynamic_node
                    node_area_G = nx.subgraph(env, node_area_nodes)

                    # Определяем лучший узел
                    best_nodes, _ = self.best_point_function(env=node_area_G, 
                                                                    start_node=dynamic_node_id,
                                                                    # area=area,
                                                                    appr_val = self.appr_val,
                                                                    **kwargs)[:2] # [:2] Это для ограничения вывода дебаг-данных в некоторых функциях
                    if isinstance(best_nodes, list):
                        best_nodes = best_nodes[0]
                    # Добавляем полученный узел в новый список
                    new_dynamic_nodes[best_nodes] = dynamic_node_key

            # Заменяем список dynamic_nodes списком с новыми, лучшими узлами
            dynamic_nodes = new_dynamic_nodes

            # Приведение к типу переменной в соответствии с типом `dynamic_nodes`
            if isinstance(dynamic_nodes,list):
                start_nodes = dynamic_nodes+static_nodes
            elif isinstance(dynamic_nodes,dict):
                start_nodes = {**dynamic_nodes, **static_nodes}

            # Расчет состояния
            times, nearest = self.state_function(env=env, points=start_nodes, **kwargs)   # area=area, 

            # Расчет метрики состояния
            state_metric = self.metric_function(times, area=area, **kwargs)
            # if best_metric is None:
            #     best_metric = state_metric

            # Оценка метрики состояния
            if self.metric_function.compare(best_metric, state_metric) == state_metric:
                best_metric = state_metric
                best_dynamic_nodes = dynamic_nodes

            # Выполняем функцию завершения расчета для итерации
            if iter_calc_end_function:
                iter_calc_end_function(i=iteration,
                                       best_metric=best_metric,
                                       dynamic_nodes=best_dynamic_nodes,
                                       static_nodes=static_nodes)

        return best_dynamic_nodes, best_metric


cmap1 = LinearSegmentedColormap.from_list("mycmap", ["green", "gold", "red"])

def iter_end_func(**kwds):
    # print('Итерация:', kwds['i'])
    print(kwds)
    dynamic_nodes = kwds['dynamic_nodes']
    if isinstance(dynamic_nodes, dict):
        dynamic_nodes_id = list(dynamic_nodes.keys())
    else:
        dynamic_nodes_id = dynamic_nodes

    try:
        static_nodes = kwds['static_nodes']
        if isinstance(static_nodes, dict):
            static_nodes_id = list(static_nodes.keys())
        else:
            static_nodes_id = static_nodes
    except:
        static_nodes = None
        
    if not static_nodes is None:
        if isinstance(dynamic_nodes,list):
            start_nodes = dynamic_nodes+static_nodes
        elif isinstance(dynamic_nodes,dict):
            start_nodes = {**dynamic_nodes, **static_nodes}
    times, nearest = FirstArrivalUnitState()(env=G, points=start_nodes)
    # times = {k:int(t%5)*5 for k, t in times.items()}

    nx.set_node_attributes(G, pd.Series(nearest, dtype=str), 'nearest')
    nx.set_node_attributes(G, times, 'route_time')

    nds = ox.graph_to_gdfs(G, edges=False)
    fig, (ax1, ax2) = plt.subplots(1,2, figsize=(12,6))
    ox.plot_graph(G, edge_linewidth=0.2, node_size=0, show=False, ax=ax1)
    nds.plot(ax=ax1, markersize=2, column='nearest', cmap='Set3', legend=True)
    nds.loc[dynamic_nodes_id].plot(ax=ax1, markersize=25, color='red')
    if static_nodes:
        nds.loc[static_nodes_id].plot(ax=ax1, markersize=25, color='k')

    ox.plot_graph(G, edge_linewidth=0.2, node_size=0, show=False, ax=ax2)
    nds.plot(ax=ax2, markersize=2, column='route_time', cmap=cmap1, legend=True)
    nds.loc[dynamic_nodes_id].plot(ax=ax2, markersize=25, color='red')
    if static_nodes:
        nds.loc[static_nodes_id].plot(ax=ax2, markersize=25, color='k')

    plt.show()





# Определение маски расчета
nodes = ox.graph_to_gdfs(G, edges=False)
points_mask = nodes.within(area.iloc[0].geometry)


nodes = list(G.nodes())
# existed_units = {nodes[100]:'A', 
#                 nodes[2000]:'B', 
#                 }
# new_units = {nodes[3000]:'c',
#              nodes[4000]:'d'}
new_units = {nodes[100]:'A', 
            nodes[2000]:'B',
            nodes[3000]:'c',
            nodes[4000]:'d'}
# existed_units = list(existed_units.keys())
# new_units = list(new_units.keys())
# print(existed_units)

# iter_end_func(dynamic_nodes=new_units, static_nodes={})


BNF = BestNodeHillClimbing(state_function=FirstArrivalUnitState(),
                       metric_function=ArrivalTime())
def BNF(env, start_node, area=None, appr_val: float = 0.95):
    bnch_max = BestNodeHillClimbing(state_function=FirstArrivalUnitState(), metric_function=ArrivalTime(np.max), appr_val=appr_val)
    bnch_mean = BestNodeHillClimbing(state_function=FirstArrivalUnitState(), metric_function=ArrivalTime(), appr_val=appr_val)

    best_node, best_metric = bnch_max(env=env, start_node=start_node, area=area)
    best_node, best_metric = bnch_mean(env=env, start_node=best_node, area=area)
    return best_node, best_metric

BNG = timing(BestNodesGenesis(state_function=FirstArrivalUnitState(),
                       metric_function=ArrivalTime(),
                       best_point_function=BNF,
                    #    best_point_function=BestNodeHillClimbingMaxMean,
                       appr_val=1,  # 0.95
                       iterations=5,
                       ))

# BNG(env=G, dynamic_nodes=new_units, static_nodes=existed_units, iter_calc_end_function=iter_end_func, area=points_mask)
# BNG(env=G, dynamic_nodes=new_units, static_nodes=existed_units, iter_calc_end_function=iter_end_func)
BNG(env=G,
    dynamic_nodes=new_units,
    # static_nodes=existed_units,
    before_iters_start_function=iter_end_func,
    iter_calc_end_function=iter_end_func,
    area=points_mask,
    )
# BNG(env=G, dynamic_nodes=new_units, iter_calc_end_function=iter_end_func, area=points_mask)

In [ ]:
BNG(env=G,
    dynamic_nodes=new_units,
    before_iters_start_function=iter_end_func,
    iter_calc_end_function=iter_end_func,
    )

In [ ]:
existed_units = {nodes[3000]:'A', 
                nodes[100]:'B', 
                }
new_units = {nodes[4000]:'c',
             nodes[2000]:'d'}


# BNF = BestNodeHillClimbing(state_function=FirstArrivalUnitState(),
#                        metric_function=ArrivalTime())
def BNF(env, start_node, area=None, appr_val: float = 0.95):
    bnch_max = BestNodeHillClimbing(state_function=FirstArrivalUnitState(), metric_function=ArrivalTime(np.max), appr_val=appr_val)
    bnch_mean = BestNodeHillClimbing(state_function=FirstArrivalUnitState(), metric_function=ArrivalTime(), appr_val=appr_val)

    best_node, best_metric = bnch_max(env=env, start_node=start_node, area=area)
    best_node, best_metric = bnch_mean(env=env, start_node=best_node, area=area)
    return best_node, best_metric

BNG = timing(BestNodesGenesis(state_function=FirstArrivalUnitState(),
                       metric_function=ArrivalTime(),
                       best_point_function=BNF,
                    #    best_point_function=BestNodeHillClimbingMaxMean,
                       appr_val=1,  # 0.95
                       iterations=5,
                       ))

BNG(env=G,
    dynamic_nodes=new_units,
    static_nodes=existed_units,
    before_iters_start_function=iter_end_func,
    iter_calc_end_function=iter_end_func,
    # area=points_mask,
    )

In [ ]:
existed_units = {nodes[3000]:'A', 
                nodes[100]:'B', 
                }
new_units = {nodes[4000]:'c',
             nodes[2000]:'d'}


# BNF = BestNodeHillClimbing(state_function=FirstArrivalUnitState(),
#                        metric_function=ArrivalTime())
def BNF(env, start_node, area=None, appr_val: float = 0.95):
    bnch_max = BestNodeHillClimbing(state_function=FirstArrivalUnitState(), metric_function=ArrivalTime(np.max), appr_val=appr_val)
    bnch_mean = BestNodeHillClimbing(state_function=FirstArrivalUnitState(), metric_function=ArrivalTime(), appr_val=appr_val)

    best_node, best_metric = bnch_max(env=env, start_node=start_node, area=area)
    best_node, best_metric = bnch_mean(env=env, start_node=best_node, area=area)
    return best_node, best_metric

BNG = timing(BestNodesGenesis(state_function=FirstArrivalUnitState(),
                       metric_function=ArrivalTime(),
                       best_point_function=BNF,
                    #    best_point_function=BestNodeHillClimbingMaxMean,
                       appr_val=1,  # 0.95
                       iterations=5,
                       ))


BNG(env=G,
    dynamic_nodes=new_units,
    static_nodes=existed_units,
    before_iters_start_function=iter_end_func,
    iter_calc_end_function=iter_end_func,
    area=points_mask,
    )